In [1]:
!pip install -q --upgrade pip
!pip install -q transformers soundfile torchaudio scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 141.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 144.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [nvidia-cusolver-cu12]


In [2]:
!pip install transformers torchaudio scipy soundfile


In [3]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [gradio]


In [4]:
!pip install transformers torchaudio soundfile scipy datasets


In [5]:
!pip install transformers==4.31.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 94.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.3
    Uninstalling transformers-4.51.3:
      Successfully uninstalled transformers-4.51.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.31.0 which is incompatible.


In [7]:
from transformers import (
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan
)
import torch
import pandas as pd
import numpy as np
import soundfile as sf
import io
import os
import time
from IPython.display import Audio, display

# ✅ Load TTS model and vocoder
processor = SpeechT5Processor.from_pretrained("MBZUAI/speecht5_tts_clartts_ar")
model = SpeechT5ForTextToSpeech.from_pretrained("MBZUAI/speecht5_tts_clartts_ar")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
model.eval()

# ✅ Step 1: Load audio sample from SADA parquet
audio_df = pd.read_parquet("/content/0017.parquet")  # Local audio parquet
audio_bytes = audio_df.loc[0, "audio"]["bytes"]
with io.BytesIO(audio_bytes) as f:
    audio_data, sr = sf.read(f)
sf.write("speaker_audio.wav", audio_data, sr)  # Optional: save for inspection

# ✅ Step 2: Load speaker embedding
xvector_df = pd.read_parquet("/content/0000.parquet")
speaker_embedding = torch.tensor(xvector_df.loc[0, "speaker_embeddings"]).unsqueeze(0).float()
print("✅ Speaker embedding shape:", speaker_embedding.shape)

# ✅ Step 3: Define test Arabic sentences
test_sentences = {
    "MSA": "مرحبًا، هذه السيارة مزودة بمحرك توربو سعة 2.0 لتر ونظام ملاحة متقدم.",
    "Najdi": "مرحبا، السيارة هاي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن متطور.",
    "Hijazi": "هلا، السيارة دي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن زين.",
    "Gulf": "هلا، السيارة ذي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن ممتاز."
}

# ✅ Step 4: Synthesize with metrics
os.makedirs("speecht5_outputs", exist_ok=True)
metrics = []

for dialect, sentence in test_sentences.items():
    print(f"\n🎙️ Generating for: {dialect}")
    inputs = processor(text=sentence, return_tensors="pt")

    start_time = time.time()
    with torch.no_grad():
        speech = model.generate_speech(
            input_ids=inputs["input_ids"],
            #attention_mask=inputs["attention_mask"],
            speaker_embeddings=speaker_embedding,
            vocoder=vocoder
        )
    end_time = time.time()

    # Save audio
    out_path = f"speecht5_outputs/{dialect}.wav"
    sf.write(out_path, speech.numpy(), samplerate=16000)
    display(Audio(out_path))

    # Evaluate
    latency = end_time - start_time
    duration = speech.shape[0] / 16000.0

    metrics.append({
        "Dialect": dialect,
        "Latency (s)": round(latency, 2),
        "Audio Duration (s)": round(duration, 2),
        "File": out_path
    })

# ✅ Step 5: Summary table
df_metrics = pd.DataFrame(metrics)
print("\n📊 Evaluation Summary:")
display(df_metrics)
df_metrics.to_csv("speecht5_outputs/eval_metrics.csv", index=False)
print("\n📊 Evaluation metrics saved to speecht5_outputs/eval_metrics.csv")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Speaker embedding shape: torch.Size([1, 512])

🎙️ Generating for: MSA



🎙️ Generating for: Najdi



🎙️ Generating for: Hijazi



🎙️ Generating for: Gulf



📊 Evaluation Summary:


,Dialect,Latency (s),Audio Duration (s),File
0,MSA,22.73,7.33,speecht5_outputs/MSA.wav
1,Najdi,14.21,7.01,speecht5_outputs/Najdi.wav
2,Hijazi,15.80,5.44,speecht5_outputs/Hijazi.wav
3,Gulf,11.70,5.76,speecht5_outputs/Gulf.wav



📊 Evaluation metrics saved to speecht5_outputs/eval_metrics.csv


In [8]:
import gradio as gd
import torch
import pandas as pd
import soundfile as sf
import time
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

# Load models once
processor = SpeechT5Processor.from_pretrained("MBZUAI/speecht5_tts_clartts_ar")
model = SpeechT5ForTextToSpeech.from_pretrained("MBZUAI/speecht5_tts_clartts_ar")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
model.eval()


# Load speaker embeddings from parquet (adjust paths if needed)
xvector_df = pd.read_parquet("/content/0000.parquet")
# Assuming one speaker embedding for simplicity, you can map dialects if needed
speaker_embedding = torch.tensor(xvector_df.loc[0, "speaker_embeddings"]).unsqueeze(0).float()

test_sentences = {
    "MSA": "مرحبًا، هذه السيارة مزودة بمحرك توربو سعة 2.0 لتر ونظام ملاحة متقدم.",
    "Najdi": "مرحبا، السيارة هاي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن متطور.",
    "Hijazi": "هلا، السيارة دي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن زين.",
    "Gulf": "هلا، السيارة ذي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن ممتاز."
}

def tts_infer(text, dialect):
    if dialect not in test_sentences:
        return None
    # Optional: override text with preset sentence for dialect
    # Or use user input text directly - here we just use input text
    inputs = processor(text=text, return_tensors="pt")
    start_time = time.time()
    with torch.no_grad():
        speech = model.generate_speech(
            input_ids=inputs["input_ids"],
            speaker_embeddings=speaker_embedding,
            vocoder=vocoder
        )
    latency = time.time() - start_time

    wav = speech.cpu().numpy()
    output_path = f"speecht5_outputs/{dialect}_{int(time.time())}.wav"
    sf.write(output_path, wav, samplerate=16000)
    print(f"Synthesized {dialect} in {latency:.2f} seconds")
    return output_path

iface = gd.Interface(
    fn=tts_infer,
    inputs=[
        gd.Textbox(label="Input Arabic Text", lines=2, placeholder="Type your text here..."),
        gd.Dropdown(choices=["MSA", "Najdi", "Hijazi", "Gulf"], label="Dialect")
    ],
    outputs=gd.Audio(type="filepath", label="Synthesized Speech"),
    title="SpeechT5 Arabic TTS Demo",
    description="Enter Arabic text and select dialect to synthesize speech using SpeechT5 with a precomputed speaker embedding."
)

iface.launch(debug=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c60527e0b944ab40b3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Synthesized MSA in 26.68 seconds
Synthesized Najdi in 14.72 seconds
Synthesized Hijazi in 13.09 seconds
Synthesized Gulf in 11.65 seconds
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c60527e0b944ab40b3.gradio.live


In [ ]:
import gradio as gd
import torch
import pandas as pd
import soundfile as sf
import time
import io
import os
from transformers import (
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
    SpeechT5SpeakerEncoder
)

os.makedirs("speecht5_outputs", exist_ok=True)

# Load models once
processor = SpeechT5Processor.from_pretrained("MBZUAI/speecht5_tts_clartts_ar")
model = SpeechT5ForTextToSpeech.from_pretrained("MBZUAI/speecht5_tts_clartts_ar")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
speaker_encoder = SpeechT5SpeakerEncoder.from_pretrained("microsoft/speecht5_speaker_encoder")

model.eval()
speaker_encoder.eval()

# Load audio for embedding extraction
audio_df = pd.read_parquet("/content/0017.parquet")
audio_bytes = audio_df.loc[0, "audio"]["bytes"]
with io.BytesIO(audio_bytes) as f:
    audio_data, sr = sf.read(f)
audio_tensor = torch.from_numpy(audio_data).unsqueeze(0)

# Load precomputed speaker embedding from vector parquet
vector_df = pd.read_parquet("/content/0000.parquet")
precomputed_embedding = torch.tensor(vector_df.loc[0, "speaker_embeddings"]).unsqueeze(0).float()

test_sentences = {
    "MSA": "مرحبًا، هذه السيارة مزودة بمحرك توربو سعة 2.0 لتر ونظام ملاحة متقدم.",
    "Najdi": "مرحبا، السيارة هاي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن متطور.",
    "Hijazi": "هلا، السيارة دي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن زين.",
    "Gulf": "هلا، السيارة ذي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن ممتاز."
}

def tts_infer(text, dialect, embedding_source):
    inputs = processor(text=text, return_tensors="pt")

    if embedding_source == "Precomputed Embedding":
        speaker_embedding = precomputed_embedding
    else:
        with torch.no_grad():
            speaker_embedding = speaker_encoder(audio_tensor)

    start_time = time.time()
    with torch.no_grad():
        speech = model.generate_speech(
            input_ids=inputs["input_ids"],
            speaker_embeddings=speaker_embedding,
            vocoder=vocoder
        )
    latency = time.time() - start_time

    wav = speech.cpu().numpy()
    output_path = f"speecht5_outputs/{dialect}_{int(time.time())}.wav"
    sf.write(output_path, wav, samplerate=16000)

    print(f"Synthesized {dialect} using {embedding_source} in {latency:.2f} seconds")
    return output_path

iface = gd.Interface(
    fn=tts_infer,
    inputs=[
        gd.Textbox(label="Input Arabic Text", lines=2, placeholder="Type your text here..."),
        gd.Dropdown(choices=["MSA", "Najdi", "Hijazi", "Gulf"], label="Dialect"),
        gd.Radio(choices=["Precomputed Embedding", "Extract from Audio"], label="Speaker Embedding Source", value="Precomputed Embedding")
    ],
    outputs=gd.Audio(type="filepath", label="Synthesized Speech"),
    title="SpeechT5 Arabic TTS Demo",
    description="Enter Arabic text and select dialect to synthesize speech using SpeechT5. Choose speaker embedding source to trade off latency and flexibility."
)

iface.launch(debug=True)
